# VoxBind generated samples — 3D visualization (nglview)

Visualizes the molecules generated by
`voxbind_frozenenc_atomblob7_v2p1` (epoch 71 snapshot) inside their target
pockets.

Snapshot: `voxbind/exps/voxbind_frozenenc_atomblob7_v2p1_SAMPLESNAP/samples/res_ep71`
(4 samples/pocket × 2 pockets: `target_02` = GRK4, `target_03` = GSTP1).

Each target dir holds:
* `samples.sdf` — the generated ligands
* `*_pocket10.pdb` — the 10Å receptor pocket
* `*_lig_*.sdf` — the reference (ground-truth) ligand

The notebook shows a 2D RDKit grid (always works) and an interactive nglview 3D
overlay of the pocket + generated ligands + reference ligand.

In [ ]:
# --- setup: imports (installs nglview on first run if missing) ---
import os, sys, glob, subprocess

try:
    import nglview as nv
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'nglview'], check=True)
    import nglview as nv

from rdkit import Chem, RDLogger
from rdkit.Chem import Draw, AllChem
from rdkit.Chem.Draw import rdMolDraw2D
RDLogger.DisableLog('rdApp.*')
print('nglview', nv.__version__, '| rdkit', Chem.rdBase.rdkitVersion)

In [ ]:
# --- locate the sample snapshot (portable across the /home1 and /home mounts) ---
CANDIDATES = [
    '/home1/irteam/VoxBind/voxbind/exps/voxbind_frozenenc_atomblob7_v2p1_SAMPLESNAP/samples/res_ep71',
    '/home/irteam/VoxBind/voxbind/exps/voxbind_frozenenc_atomblob7_v2p1_SAMPLESNAP/samples/res_ep71',
]
SNAP = next((p for p in CANDIDATES if os.path.isdir(p)), None)
if SNAP is None:
    # fall back: walk up from CWD to the repo root, then into the exp dir
    r = os.getcwd()
    while r != os.path.dirname(r) and not os.path.isdir(os.path.join(r, 'voxbind', 'exps')):
        r = os.path.dirname(r)
    SNAP = os.path.join(r, 'voxbind', 'exps',
                        'voxbind_frozenenc_atomblob7_v2p1_SAMPLESNAP', 'samples', 'res_ep71')
assert os.path.isdir(SNAP), f'snapshot dir not found: {SNAP}'

TARGETS = sorted(glob.glob(os.path.join(SNAP, 'target_*')))
print('snapshot :', SNAP)
print('targets  :', [os.path.basename(t) for t in TARGETS])

In [ ]:
# --- per-target loaders ---
def target_files(tdir):
    """Return (pocket_pdb, ref_ligand_sdf, samples_sdf) for a target dir."""
    pocket = glob.glob(os.path.join(tdir, '*_pocket10.pdb'))[0]
    samples = os.path.join(tdir, 'samples.sdf')
    refs = [f for f in glob.glob(os.path.join(tdir, '*.sdf'))
            if os.path.basename(f) != 'samples.sdf']
    return pocket, (refs[0] if refs else None), samples

def load_mols(sdf):
    """Valid, connected molecules from an SDF (keeps conformers)."""
    out = []
    if not sdf or not os.path.exists(sdf):
        return out
    for m in Chem.SDMolSupplier(sdf, removeHs=False, sanitize=True):
        if m is None:
            continue
        try:
            if '.' not in Chem.MolToSmiles(m):
                out.append(m)
        except Exception:
            pass
    return out

for t in TARGETS:
    pocket, ref, samples = target_files(t)
    print(f"{os.path.basename(t):10s}  samples={len(load_mols(samples))}  "
          f"ref={'yes' if ref else 'no':3s}  pocket={os.path.basename(pocket)}")

## 2D overview (RDKit)
A quick topology grid of the generated ligands — renders even without a WebGL
backend, so it is a good sanity check before the 3D view.

In [ ]:
from IPython.display import display

for t in TARGETS:
    _, _, samples = target_files(t)
    mols = load_mols(samples)
    flat = []
    for m in mols:
        m2 = Chem.Mol(m)
        try:
            AllChem.Compute2DCoords(m2)
        except Exception:
            pass
        flat.append(m2)
    print(os.path.basename(t))
    display(Draw.MolsToGridImage(
        flat, molsPerRow=4, subImgSize=(260, 220),
        legends=[f'sample {i}' for i in range(len(flat))]))

## 3D pocket + ligands (nglview)
Pocket drawn as cartoon + transparent surface. Generated ligands are licorice in
distinct colors; the reference ligand is thick green licorice. Rotate/zoom with
the mouse.

In [ ]:
import tempfile

# distinct colors for generated ligands
PALETTE = ['orange', 'magenta', 'cyan', 'yellow', 'salmon', 'purple',
           'lime', 'pink', 'skyblue', 'red']

def _add_mol(view, mol, color, radius=0.25):
    """Add one RDKit mol (with 3D conformer) as a licorice component."""
    tf = tempfile.NamedTemporaryFile(suffix='.sdf', delete=False)
    w = Chem.SDWriter(tf.name); w.write(mol); w.close()
    comp = view.add_component(tf.name)
    comp.clear_representations()
    comp.add_licorice(color=color, radius=radius)
    return comp

def view_target(tdir, show_ref=True, show_surface=True):
    pocket, ref, samples = target_files(tdir)
    view = nv.NGLWidget()
    # receptor pocket
    prot = view.add_component(pocket)
    prot.clear_representations()
    prot.add_cartoon(color='sstruc')
    prot.add_licorice('protein', opacity=0.35, radius=0.12)
    if show_surface:
        prot.add_surface(color='white', opacity=0.12, wireframe=False)
    # reference ligand (green)
    if show_ref and ref:
        for rm in load_mols(ref):
            _add_mol(view, rm, 'green', radius=0.3)
    # generated ligands
    for i, m in enumerate(load_mols(samples)):
        _add_mol(view, m, PALETTE[i % len(PALETTE)])
    view.center()
    view._remote_call('setSize', target='Widget', args=['100%', '520px'])
    return view

print('reference ligand = GREEN | generated samples =', ', '.join(PALETTE[:4]))

In [ ]:
# target_02 — GRK4
view_target(TARGETS[0])

In [ ]:
# target_03 — GSTP1
view_target(TARGETS[1])

### Optional: isolate one sample
Use the dropdown to overlay the pocket with a single generated ligand plus the
reference, e.g. to inspect a specific high-affinity pose from the docking eval.

In [ ]:
import ipywidgets as widgets

def show_single(target_idx, sample_idx):
    tdir = TARGETS[target_idx]
    pocket, ref, samples = target_files(tdir)
    mols = load_mols(samples)
    if not mols:
        print('no samples'); return
    sample_idx = min(sample_idx, len(mols) - 1)
    view = nv.NGLWidget()
    prot = view.add_component(pocket)
    prot.clear_representations()
    prot.add_cartoon(color='sstruc')
    prot.add_surface(color='white', opacity=0.12)
    if ref:
        for rm in load_mols(ref):
            _add_mol(view, rm, 'green', radius=0.3)
    _add_mol(view, mols[sample_idx], 'orange', radius=0.3)
    view.center()
    view._remote_call('setSize', target='Widget', args=['100%', '520px'])
    display(view)

widgets.interact(
    show_single,
    target_idx=widgets.Dropdown(
        options=[(os.path.basename(t), i) for i, t in enumerate(TARGETS)], value=0,
        description='target'),
    sample_idx=widgets.IntSlider(min=0, max=3, value=0, description='sample'));